# 01 · Classes and objects

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 85 minutes live (with a coffee break in the middle) | [00 · Warm-up](00_warmup_whats_wrong.ipynb) | [02 · Inheritance](02_inheritance.ipynb) |

In the warm-up notebook, everything lived in loose variables: `df`, `df2`, `le`, `X_train`, `preds`, ... Nothing told you which of them belonged together, and nothing stopped you from mixing them up. A **class** lets you bundle data and the functions that work on that data into one named thing. Almost every library you use (pandas, scikit-learn, PyTorch) is built from classes, so understanding them is also the key to reading other people's code.

> **Learning goals.** By the end you can:
>
> - explain the difference between a class and an object (instance) in your own words
> - write a class with `__init__`, attributes and methods, and create several independent objects from it
> - decide when a derived value should be a `@property` and recognise `@classmethod` constructors in library code
> - spot the shared-mutable class attribute pitfall

**Before you start.** Notebook 00. Python basics (P0) are enough; no prior OOP needed.

**Exercises** come in three levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for those who already write classes). Check cells fail until your code is right; that is expected.

**In this notebook**

1. The problem: loose variables
2. Classes and objects
3. `__init__` and `self`: giving objects their data
4. Methods: behaviour that belongs to the data
5. Class attributes and a classic pitfall
6. Properties: derived values that look like attributes
7. `@classmethod`: another way to build an object
8. Exercises: a `ChurnDataset` class

---

## 1. The problem: loose variables

We need some predictions to work with. The details of the model do not matter for this notebook: we fit a simple logistic regression on four numeric columns and ask for each customer's churn probability.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH)
# TotalCharges is text and blank for customers who joined this month (tenure 0): they owe nothing yet
customers["TotalCharges"] = pd.to_numeric(customers["TotalCharges"], errors="coerce").fillna(0.0)
customers["Churn"] = (customers["Churn"] == "Yes").astype(int)

features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
X_train, X_test, y_train, y_test = train_test_split(
    customers[features], customers["Churn"], test_size=0.2, random_state=42
)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
proba[:5]

The retention team asks: "if we call every customer with a churn probability above 0.5, how many churners do we reach? And what if we lower the bar to 0.3?" To answer, we count hits and misses (a **confusion matrix**) for each threshold.

Without classes, the code looks like this:

In [ ]:
y_true = y_test.to_numpy()

pred_50 = (proba >= 0.5).astype(int)
tp_50 = int(((y_true == 1) & (pred_50 == 1)).sum())
fp_50 = int(((y_true == 0) & (pred_50 == 1)).sum())
fn_50 = int(((y_true == 1) & (pred_50 == 0)).sum())
tn_50 = int(((y_true == 0) & (pred_50 == 0)).sum())
recall_50 = tp_50 / (tp_50 + fn_50)
precision_50 = tp_50 / (tp_50 + fp_50)

pred_30 = (proba >= 0.3).astype(int)
tp_30 = int(((y_true == 1) & (pred_30 == 1)).sum())
fp_30 = int(((y_true == 0) & (pred_30 == 1)).sum())
fn_30 = int(((y_true == 1) & (pred_30 == 0)).sum())
tn_30 = int(((y_true == 0) & (pred_30 == 0)).sum())
recall_30 = tp_30 / (tp_30 + fn_30)
precision_30 = tp_30 / (tp_30 + fp_30)

print(f"threshold 0.5: precision={precision_50:.2f} recall={recall_50:.2f}")
print(f"threshold 0.3: precision={precision_30:.2f} recall={recall_30:.2f}")

Fourteen variables for two thresholds, the same formulas typed twice, and nothing tells you that `tp_50` and `fn_50` belong together while `tp_50` and `fn_30` do not. A third threshold means another seven lines. One typo (`tp_50` where you meant `tp_30`) and the numbers are silently wrong.

What we want is one **thing** called "confusion matrix" that holds its four counts and knows how to compute its own precision and recall.

---

## 2. Classes and objects

A **class** is a blueprint. It describes what data a kind of thing holds (its **attributes**) and what it can do (its **methods**). An **object** (also called an **instance**) is one concrete thing built from that blueprint.

You already use this all the time: `pd.DataFrame` is a class, and every DataFrame you create is an object of that class. `LogisticRegression` is a class, and `model` above holds one object of it.

In [ ]:
print(type(customers))
print(type(model))
print(isinstance(customers, pd.DataFrame))

The smallest possible class:

In [ ]:
class ConfusionMatrix:
    pass


cm_a = ConfusionMatrix()  # calling the class creates a new object
cm_b = ConfusionMatrix()
print(cm_a)
print(cm_a is cm_b)  # two separate objects in memory

---

## 3. `__init__` and `self`: giving objects their data

An empty object is not useful. We want every confusion matrix to start life with its four counts. Python calls a special method named `__init__` automatically when an object is created, and passes the arguments you gave to the class.

In [ ]:
class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp


cm = ConfusionMatrix(tn=900, fp=135, fn=170, tp=204)
print(cm.tp, cm.fn)

What is `self`? It is the object being created (or, later, the object a method is called on). `ConfusionMatrix(tn=900, ...)` roughly does this:

1. create a new, empty object;
2. call `__init__(that_new_object, tn=900, fp=135, fn=170, tp=204)`;
3. inside `__init__`, `self.tp = tp` stores the value **on that object**;
4. hand the object back to you.

`self` is just the name of the first parameter. You could call it anything, but everyone uses `self`, so do too.

Values stored as `self.something` are **instance attributes**: each object has its own.

---

## 4. Methods: behaviour that belongs to the data

A **method** is a function defined inside a class. Its first parameter is `self`, so it can read the object's attributes. You call it with a dot: `cm.recall()`. Python fills in `self` for you.

In [ ]:
class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    def precision(self) -> float:
        """Of the customers we flagged, how many really churned?"""
        return self.tp / (self.tp + self.fp)

    def recall(self) -> float:
        """Of the customers who churned, how many did we flag?"""
        return self.tp / (self.tp + self.fn)

    def f1(self) -> float:
        p, r = self.precision(), self.recall()  # methods can call other methods via self
        return 2 * p * r / (p + r)


cm = ConfusionMatrix(tn=900, fp=135, fn=170, tp=204)
print(f"precision={cm.precision():.2f} recall={cm.recall():.2f} f1={cm.f1():.2f}")

`cm.recall()` is a shortcut for `ConfusionMatrix.recall(cm)`. Both work; the first is what everyone writes.

### Same class, different objects

Now the two thresholds become two objects. Each carries its own counts, and the formulas exist exactly once.

In [ ]:
def count(y_true, y_pred) -> ConfusionMatrix:
    return ConfusionMatrix(
        tn=int(((y_true == 0) & (y_pred == 0)).sum()),
        fp=int(((y_true == 0) & (y_pred == 1)).sum()),
        fn=int(((y_true == 1) & (y_pred == 0)).sum()),
        tp=int(((y_true == 1) & (y_pred == 1)).sum()),
    )


results = {t: count(y_true, (proba >= t).astype(int)) for t in [0.5, 0.4, 0.3, 0.2]}
for threshold, cm in results.items():
    print(
        f"threshold {threshold}: flagged={cm.tp + cm.fp:4d}  precision={cm.precision():.2f}  recall={cm.recall():.2f}"
    )

A lower threshold reaches more churners (higher recall) but also calls more customers who would have stayed anyway (lower precision). Which threshold is right is a business decision; we come back to it with a cost model in notebook 03.

> **Check your understanding**
>
> ```python
> a = ConfusionMatrix(tn=10, fp=0, fn=5, tp=5)
> b = a
> b.tp = 10
> print(a.recall())
> ```
>
> What is printed: `0.5`, `0.666...`, or an error?
>
> <details><summary><b>Show answer</b></summary>
>
> `0.666...` (that is 10 / 15). `b = a` does not copy the object; it gives the same object a second name. Changing `b.tp` changes the one and only object, which `a` also refers to. To get two independent objects you must create two: `ConfusionMatrix(...)` twice.
> </details>

### Exercise 1.1
> **Core** · for everyone

Add a method `accuracy()` (share of all customers classified correctly) and a method `n()` (total number of customers) to the class below. Then check: at threshold 0.5, is accuracy a good summary for the retention team?

In [ ]:
# Exercise 1.1: your code here


class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    def precision(self) -> float:
        return self.tp / (self.tp + self.fp)

    def recall(self) -> float:
        return self.tp / (self.tp + self.fn)

    def n(self) -> int:
        raise NotImplementedError("Your code here")

    def accuracy(self) -> float:
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 1.1: run after solving it
cm = ConfusionMatrix(tn=900, fp=135, fn=170, tp=204)
assert cm.n() == 1409
assert abs(cm.accuracy() - 1104 / 1409) < 1e-9
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    def precision(self) -> float:
        return self.tp / (self.tp + self.fp)

    def recall(self) -> float:
        return self.tp / (self.tp + self.fn)

    def n(self) -> int:
        return self.tn + self.fp + self.fn + self.tp

    def accuracy(self) -> float:
        return (self.tp + self.tn) / self.n()
```

About 73 % of customers do not churn, so a "model" that predicts "nobody churns" already reaches 73 % accuracy while finding zero churners. Recall and precision tell the retention team much more.
</details>

---

## 5. Class attributes and a classic pitfall

Attributes set on `self` belong to one object. Attributes written directly in the class body belong to the **class** and are shared by all its objects. They are useful for constants that are the same for every object.

In [ ]:
class ConfusionMatrix:
    positive_label = "churn"  # class attribute: the same for every confusion matrix

    def __init__(self, tn, fp, fn, tp):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp  # instance attributes


a = ConfusionMatrix(1, 2, 3, 4)
b = ConfusionMatrix(5, 6, 7, 8)
print(a.positive_label, b.positive_label, ConfusionMatrix.positive_label)
print(a.tp, b.tp)

> **Check your understanding**
>
> A colleague writes a small class to keep a log of experiments:
>
> ```python
> class ExperimentLog:
>     runs = []
>
>     def __init__(self, name):
>         self.name = name
>
>     def add(self, metric_value):
>         self.runs.append(metric_value)
>
>
> baseline = ExperimentLog("baseline")
> forest = ExperimentLog("random forest")
> baseline.add(0.61)
> forest.add(0.64)
> print(baseline.runs)
> ```
>
> What is printed?
>
> <details><summary><b>Show answer</b></summary>
>
> `[0.61, 0.64]`. `runs` is a class attribute, so there is a single list shared by **all** logs. `self.runs.append(...)` does not create a new list on the object; it finds the class's list and modifies it. The forest's result ends up in the baseline's log. The fix: create the list in `__init__` with `self.runs = []`, so every object gets its own. Rule of thumb: mutable things (lists, dicts, DataFrames) belong in `__init__`, not in the class body.
> </details>

---

## 6. Properties: derived values that look like attributes

`cm.recall()` needs parentheses, `cm.tp` does not. But both are just facts about the matrix. A `@property` turns a method without arguments into something you read like an attribute. It is still computed on every access, so it can never be out of date.

In [ ]:
class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp

    @property
    def n(self) -> int:
        return self.tn + self.fp + self.fn + self.tp

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp)

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn)


cm = ConfusionMatrix(tn=900, fp=135, fn=170, tp=204)
print(cm.recall)  # no parentheses
cm.tp = 250  # suppose we correct a count ...
print(cm.recall)  # ... the property is recomputed

try:
    cm.recall = 1.0
except AttributeError as exc:
    print(f"AttributeError: {exc}")

A property without a setter is read-only, which is what we want: recall follows from the counts, nobody should set it by hand.

**Property or method?** Use a property for cheap values derived from the object's own data, with no arguments (`df.shape`, `df.columns`, `cm.recall`). Use a method when it takes arguments, takes noticeable time, or changes something (`df.groupby(...)`, `model.fit(...)`).

---

## 7. `@classmethod`: another way to build an object

Our `count()` helper creates a `ConfusionMatrix` from labels and predictions. That is really a second way of constructing the class, so it belongs to the class. A `@classmethod` receives the class itself (by convention called `cls`) instead of an object, and typically returns a new object.

In [ ]:
class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        return cls(
            tn=int(((y_true == 0) & (y_pred == 0)).sum()),
            fp=int(((y_true == 0) & (y_pred == 1)).sum()),
            fn=int(((y_true == 1) & (y_pred == 0)).sum()),
            tp=int(((y_true == 1) & (y_pred == 1)).sum()),
        )

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp)

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn)


cm = ConfusionMatrix.from_predictions(y_test, model.predict(X_test))
print(cm.tn, cm.fp, cm.fn, cm.tp, round(cm.recall, 2))

You will meet this pattern constantly when reading library code: `pd.DataFrame.from_dict(...)`, `sklearn.metrics.ConfusionMatrixDisplay.from_predictions(...)`, `datetime.fromisoformat(...)`. Whenever a name starts with `from_`, expect a classmethod that builds an object from some other kind of input.

---

## 8. Exercises: a `ChurnDataset` class

In the warm-up, the data and everything derived from it (train set, test set, churn rate) were loose variables. Let's bundle them.

### Exercise 8.1
> **Core** · for everyone

Write a class `ChurnDataset` that wraps a customer DataFrame (like `customers` above, with `Churn` as 0/1):

- `__init__(self, df, target="Churn")` stores the DataFrame and the target column name, and sets `self.train` and `self.test` to `None` (good habit: create every attribute in `__init__`, so readers see the full list in one place);
- a property `n_customers`: number of rows;
- a method `churn_rate()`: share of customers with target 1;
- a method `churn_rate_by(column)`: churn rate per value of `column`, as a pandas Series (hint: `groupby`);
- a method `split(test_size=0.2, random_state=42)` that splits the DataFrame with `train_test_split` and stores the parts in `self.train` and `self.test`.

In [ ]:
# Exercise 8.1: your code here


class ChurnDataset:
    def __init__(self, df: pd.DataFrame, target: str = "Churn"):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 8.1: run after solving it
ds = ChurnDataset(customers)
assert ds.n_customers == 7043
assert round(ds.churn_rate(), 3) == 0.265
by_contract = ds.churn_rate_by("Contract")
assert round(by_contract["Two year"], 3) == 0.028
assert ds.train is None and ds.test is None
ds.split(test_size=0.25)
assert len(ds.test) == 1761 and len(ds.train) == 7043 - 1761
print(by_contract.round(3))
print("All checks passed")

<details><summary><b>Show hint</b></summary>

`churn_rate()` is the mean of the 0/1 target column. `churn_rate_by` is `self.df.groupby(column)[self.target].mean()`. In `split`, call `train_test_split(self.df, test_size=..., random_state=...)`: it returns two DataFrames when you pass one.
</details>

<details><summary><b>Show solution</b></summary>

```python
class ChurnDataset:
    def __init__(self, df: pd.DataFrame, target: str = "Churn"):
        self.df = df
        self.target = target
        self.train = None
        self.test = None

    @property
    def n_customers(self) -> int:
        return len(self.df)

    def churn_rate(self) -> float:
        return self.df[self.target].mean()

    def churn_rate_by(self, column: str) -> pd.Series:
        return self.df.groupby(column)[self.target].mean()

    def split(self, test_size: float = 0.2, random_state: int = 42) -> None:
        self.train, self.test = train_test_split(
            self.df, test_size=test_size, random_state=random_state
        )
```
</details>

### Exercise 8.2
> **Stretch** · if you finish early

Two weaknesses of the Core version:

1. Reading `ds.train` before `split()` silently gives `None`, and the error appears much later (`'NoneType' object has no attribute ...`).
2. With a random split, the test set's churn rate can differ from the full data's by chance, which makes model comparisons noisier.

Rewrite the class so that `train` and `test` are **properties** that raise a `RuntimeError` with a clear message if `split()` was not called yet (store the data in `self._train` and `self._test`; the underscore says "internal", more on that in notebook 04), and the split is **stratified** by the target (`stratify=` argument). Let `split` return `self`, so you can write `ChurnDataset(customers).split().train`.

In [ ]:
# Exercise 8.2: your code here


class ChurnDataset:
    def __init__(self, df: pd.DataFrame, target: str = "Churn"):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 8.2: run after solving it
ds = ChurnDataset(customers)
try:
    ds.train
    raise AssertionError("expected a RuntimeError before split()")
except RuntimeError as exc:
    print("Good error message:", exc)

ds = ChurnDataset(customers).split(test_size=0.2, random_state=1)
full_rate = customers["Churn"].mean()
assert abs(ds.test["Churn"].mean() - full_rate) < 0.002, "the split should be stratified"
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class ChurnDataset:
    def __init__(self, df: pd.DataFrame, target: str = "Churn"):
        self.df = df
        self.target = target
        self._train = None
        self._test = None

    @property
    def n_customers(self) -> int:
        return len(self.df)

    def churn_rate(self) -> float:
        return self.df[self.target].mean()

    def churn_rate_by(self, column: str) -> pd.Series:
        return self.df.groupby(column)[self.target].mean()

    def split(self, test_size: float = 0.2, random_state: int = 42) -> "ChurnDataset":
        self._train, self._test = train_test_split(
            self.df,
            test_size=test_size,
            random_state=random_state,
            stratify=self.df[self.target],
        )
        return self

    @property
    def train(self) -> pd.DataFrame:
        if self._train is None:
            raise RuntimeError("Call split() before accessing train.")
        return self._train

    @property
    def test(self) -> pd.DataFrame:
        if self._test is None:
            raise RuntimeError("Call split() before accessing test.")
        return self._test
```
</details>

### Exercise 8.3
> **Challenge** · for experienced programmers

Our `split()` **mutates** the object: before the call it has no train set, after it does. An alternative design never changes the object and instead **returns two new `ChurnDataset` objects**: `train_ds, test_ds = ds.split()`.

1. Implement that immutable version. Also make `__init__` validate its input: raise a `ValueError` if the target column is missing or contains anything other than 0 and 1.
2. Think about the trade-offs. Which design is easier to test? Which one can be used safely by two parts of a program at once? What does scikit-learn's `train_test_split` do, and what does a pandas method like `df.dropna()` do by default?

In [ ]:
# Exercise 8.3: your code here


class ChurnDataset:
    def __init__(self, df: pd.DataFrame, target: str = "Churn"):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 8.3: run after solving it
ds = ChurnDataset(customers)
train_ds, test_ds = ds.split(test_size=0.2, random_state=42)
assert isinstance(train_ds, ChurnDataset) and isinstance(test_ds, ChurnDataset)
assert train_ds.n_customers + test_ds.n_customers == ds.n_customers
assert ds.n_customers == 7043, "the original object must not change"
for bad in [customers.drop(columns="Churn"), customers.assign(Churn="Yes")]:
    try:
        ChurnDataset(bad)
        raise AssertionError("expected a ValueError")
    except ValueError as exc:
        print("Rejected:", exc)
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class ChurnDataset:
    def __init__(self, df: pd.DataFrame, target: str = "Churn"):
        if target not in df.columns:
            raise ValueError(f"Target column {target!r} not found")
        unexpected = set(df[target].unique()) - {0, 1}
        if unexpected:
            raise ValueError(f"Target {target!r} must be 0/1, found {sorted(map(str, unexpected))}")
        self.df = df
        self.target = target

    @property
    def n_customers(self) -> int:
        return len(self.df)

    def churn_rate(self) -> float:
        return self.df[self.target].mean()

    def churn_rate_by(self, column: str) -> pd.Series:
        return self.df.groupby(column)[self.target].mean()

    def split(
        self, test_size: float = 0.2, random_state: int = 42
    ) -> tuple["ChurnDataset", "ChurnDataset"]:
        train, test = train_test_split(
            self.df, test_size=test_size, random_state=random_state, stratify=self.df[self.target]
        )
        return ChurnDataset(train, self.target), ChurnDataset(test, self.target)
```

Some trade-offs:

- **Immutable** objects are easier to reason about and to test: an object's state never depends on which methods were called before. They can be shared without surprises. The cost is more objects and the need to pass the results around.
- **Mutable** objects can be convenient (one object holds everything), but you must know the call order (`split` before `train`) and two pieces of code holding the same object can step on each other.
- `train_test_split` returns new objects. pandas methods return new DataFrames by default too (`inplace=False`); the pandas developers even discourage `inplace=True`. scikit-learn estimators, on the other hand, are mutable on purpose: `fit` changes the model and returns `self`. Both styles exist in good libraries; what matters is choosing consciously and documenting it.
</details>

---

## Summary

- A **class** is a blueprint; an **object** (instance) is one thing built from it. `pd.DataFrame` and `LogisticRegression` are classes you already use.
- `__init__` runs when an object is created and stores its data as **instance attributes** on `self`.
- **Methods** are functions that belong to the class and work on `self`'s data. The formulas live in one place; every object reuses them.
- `@property` exposes a derived value like an attribute and keeps it consistent with the data. `@classmethod` builds objects in alternative ways (`from_predictions`).
- **Pitfalls:** `b = a` does not copy an object. Mutable values written in the class body are shared by all objects; create them in `__init__`.

## Going further

- The finished version of `ConfusionMatrix` in this repository: `src/churn/evaluation.py` (it uses a dataclass, which you will meet in notebook 07).

---

## Sources and further reading

**Sources.** This notebook is based on:

- [The Python Tutorial, chapter 9: Classes](https://docs.python.org/3/tutorial/classes.html).
- Python docs: built-in functions [`property`](https://docs.python.org/3/library/functions.html#property) and [`classmethod`](https://docs.python.org/3/library/functions.html#classmethod).
- scikit-learn's own alternative constructor: [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html).

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Python Classes, the Power of Object-Oriented Programming](https://realpython.com/python-classes/) (article, beginner): a longer walk through everything in this notebook.
- [Real Python: Python's property()](https://realpython.com/python-property/) (article, beginner to intermediate): read-only, computed and validated attributes.
- [Real Python: Instance, Class, and Static Methods Demystified](https://realpython.com/instance-class-and-static-methods-demystified/) (article, beginner).
- [Raymond Hettinger, *Python's Class Development Toolkit*](https://www.youtube.com/watch?v=HTLu2DFOdTg) (talk, PyCon 2013, 45 min): one class built step by step, with the reasoning behind each feature. Highly recommended.
- In this repository: `src/churn/evaluation.py` contains the finished `ConfusionMatrix`.

---

[← 00 · Warm-up](00_warmup_whats_wrong.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [02 · Inheritance →](02_inheritance.ipynb)